# Deploy o3-deep-research backend

**This notebook is optional.** Skip it if the core gateway deployment has already been run - the hub
Bicep from that lab already deploys the Norway East research hub and the APIM routing
policy for `o3-deep-research`.

Run this notebook only if:
- the core gateway deployment was run from an older version that did not include the Norway East hub, OR
- You want a dedicated APIM subscription key for the deep research workload.

## What this notebook does

1. Checks whether the `openai-research` APIM backend already exists.
2. If not: deploys `main.bicep` to the core resource group, creating:
   - `aif-research-{suffix}` Norway East CognitiveServices account
   - `o3-deep-research` model deployment
   - `openai-research` APIM backend + routing policy
   - `foundry-gateway-dr` APIM subscription
3. Writes `DR_MODEL` and `DR_GATEWAY_KEY` to `.env`.

## Prerequisites

- Core gateway deployed (core APIM, `GATEWAY_URL` in `.env`)
- `az login` done

## Step 1: Load configuration

In [ ]:
import json
import os
import subprocess
import base64
from pathlib import Path

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
env_file = repo_root / '.env'

with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

GATEWAY_URL = os.environ['GATEWAY_URL']
CHAT_MODEL  = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

print(f'Gateway URL  : {GATEWAY_URL}')
print(f'Chat model   : {CHAT_MODEL}')

## Step 2: Resolve core resource group and principal

In [ ]:
# Derive APIM name and suffix from GATEWAY_URL
# e.g. https://apim-foundry-{suffix}.azure-api.net/openai -> apim-foundry-{suffix}
APIM_NAME = GATEWAY_URL.split('//')[1].split('.')[0]
SUFFIX    = APIM_NAME.split('-')[-1]          # e.g. {suffix}
CORE_RG    = f'rg-foundry-core-{SUFFIX}'

# Get deployer principal ID from cached JWT
token   = subprocess.run(
    'az account get-access-token --query accessToken -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
padding = '=' * (4 - len(token.split('.')[1]) % 4)
PRINCIPAL_ID = json.loads(base64.b64decode(token.split('.')[1] + padding))['oid']

print(f'APIM service : {APIM_NAME}')
print(f'Core RG       : {CORE_RG}')
print(f'Principal ID : {PRINCIPAL_ID}')

## Step 3: Check if o3-deep-research backend already exists

In [ ]:
check = subprocess.run(
    f'az apim backend show -g "{CORE_RG}" --service-name "{APIM_NAME}" --backend-id openai-research -o none',
    shell=True, capture_output=True, text=True
)
BACKEND_EXISTS = check.returncode == 0

if BACKEND_EXISTS:
    print('✅ openai-research APIM backend already exists - skipping Bicep deployment.')
    print('   Proceeding to read existing resources.')
else:
    print('ℹ️  openai-research backend not found - will deploy main.bicep.')

## Step 4: Deploy main.bicep (if needed)

Skip this cell if the backend already exists. Takes ~5 minutes.

In [ ]:
if not BACKEND_EXISTS:
    result = subprocess.run(
        [
            'az', 'deployment', 'group', 'create',
            '-g', CORE_RG,
            '--template-file', 'main.bicep',
            '-p', f'deployerPrincipalId={PRINCIPAL_ID}',
            '-p', f'existingApimName={APIM_NAME}',
            '--name', 'dr-backend',
            '-o', 'table',
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Bicep deployment failed - see stderr above')
    print('✅ Bicep deployment complete')
else:
    print('ℹ️  Skipped (backend already exists)')

## Step 5: Resolve DR APIM subscription key

In [ ]:
# Try the dedicated deep research subscription first;
# fall back to the alpha subscription if it doesn't exist yet.
keys_result = subprocess.run(
    f'az apim subscription list-secrets -g "{CORE_RG}" --service-name "{APIM_NAME}" '
    f'--subscription-id foundry-gateway-dr --query primaryKey -o tsv',
    shell=True, capture_output=True, text=True
)

if keys_result.returncode == 0 and keys_result.stdout.strip():
    DR_GATEWAY_KEY = keys_result.stdout.strip()
    print('✅ Using foundry-gateway-dr subscription key')
else:
    # Fall back to alpha subscription (created by the core gateway deployment)
    fallback = subprocess.run(
        f'az apim subscription list-secrets -g "{CORE_RG}" --service-name "{APIM_NAME}" '
        f'--subscription-id foundry-gateway-alpha --query primaryKey -o tsv',
        shell=True, capture_output=True, text=True
    )
    if fallback.returncode == 0 and fallback.stdout.strip():
        DR_GATEWAY_KEY = fallback.stdout.strip()
        print('ℹ️  Using foundry-gateway-alpha subscription key (fallback)')
    else:
        raise RuntimeError('Could not retrieve APIM subscription key. Check az login and hub RG.')

DR_MODEL = 'o3-deep-research'

print(f'DR model     : {DR_MODEL}')
print(f'DR key       : {DR_GATEWAY_KEY[:4]}... (hidden)')

## Step 6: Write DR_* env vars to .env

In [ ]:
lines = env_file.read_text().splitlines() if env_file.exists() else []

updates = {
    'DR_MODEL':       DR_MODEL,
    'DR_GATEWAY_KEY': DR_GATEWAY_KEY,
}

# Remove existing DR_* lines, then append updated values
kept = [ln for ln in lines if not any(ln.startswith(k + '=') for k in updates)]
kept += [f'{k}={v}' for k, v in updates.items()]

env_file.write_text('\n'.join(kept) + '\n')

print('✅ .env updated:')
for k, v in updates.items():
    masked = v[:4] + '...' if len(v) > 4 and 'KEY' in k else v
    print(f'   {k}={masked}')

## Next step

Open [`12-02-deep-research-loop.ipynb`](12-02-deep-research-loop.ipynb) to run the
agentic deep research loop over the `arxiv-nlp` knowledge base.